In [0]:
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/bronze/raw/"
FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for table, fname in FILES.items():
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)   # keep everything as string in bronze
          .option("multiLine", True)
          .option("escape", '"')
          .csv(BASE + fname))
    # some CSVs carry a hidden BOM character in the first column name
    df = df.toDF(*[c.replace("\ufeff", "").strip() for c in df.columns])
    df = (df.withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_file", F.lit(fname)))
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"workspace.bronze.{table}"))
    print(table, df.count())